# תמלול הקלטות בעברית - גרסת Google Colab (חינמי, בלי להתקין כלום במחשב)

**הוראות שימוש - חשוב לקרוא לפני שמתחילים:**

1. למעלה בתפריט: **Runtime** ← **Change runtime type** ← בחרו **T4 GPU** ← **Save**.
   (עושים את זה פעם אחת - זה מאיץ את התמלול משמעותית, וזה עדיין בחינם.)
2. מריצים כל תא (cell) לפי הסדר, **מלמעלה למטה** - לוחצים על כפתור ה-▶ שמופיע
   מצד שמאל של כל תא, וממתינים שהוא יסיים (העיגול המסתובב נעלם) לפני שעוברים
   לתא הבא.
3. בתא ההעלאה תתבקשו לבחור קובץ הקלטה מהמחשב שלכם.
4. בסוף, קובצי התמלול (TXT / SRT / Word) יורדו אוטומטית לתיקיית ההורדות
   בדפדפן.

אם משהו נכשל - הריצו שוב את התא שנכשל (לפעמים זו שגיאת רשת זמנית).


## שלב 1: התקנת תוכנה (חד-פעמי בכל פעם שפותחים את הקובץ מחדש)

In [ ]:
!pip -q install faster-whisper pyannote.audio python-docx
print("הותקן בהצלחה")


## שלב 2: טוקן Hugging Face (נדרש לזיהוי דוברים)

1. הירשמו/התחברו ב-https://huggingface.co
2. צרו טוקן (הרשאת Read מספיקה) ב-https://huggingface.co/settings/tokens
3. עם אותו חשבון, אשרו תנאי שימוש בשני העמודים האלה (חובה שניהם):
   - https://huggingface.co/pyannote/speaker-diarization-3.1
   - https://huggingface.co/pyannote/segmentation-3.0

הריצו את התא הבא, הדביקו את הטוקן בתיבה שתופיע, ולחצו Enter (הטקסט לא
יוצג על המסך - זה תקין).


In [ ]:
from getpass import getpass
HF_TOKEN = getpass("הדביקו כאן את הטוקן מ-Hugging Face ולחצו Enter: ")


## שלב 3 (רשות): מפתח Anthropic - לניחוש אוטומטי של שמות דוברים

אם יש לכם מפתח API של Anthropic (מ-https://console.anthropic.com/), הדביקו
אותו בתא הבא - המערכת תנסה לזהות שמות אמיתיים של הדוברים מתוך השיחה עצמה
(אם מישהו הזדהה בשם). **אפשר גם להשאיר ריק ופשוט ללחוץ Enter** - במקרה כזה
הדוברים יסומנו "דובר 1", "דובר 2" וכו', וניתן יהיה לשנות ידנית בשלב מאוחר
יותר.


In [ ]:
from getpass import getpass
ANTHROPIC_API_KEY = getpass("מפתח Anthropic (אפשר להשאיר ריק וללחוץ Enter): ")


## שלב 4: העלאת קובץ ההקלטה

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_filename = list(uploaded.keys())[0]
print(f"הקובץ שהועלה: {audio_filename}")


## שלב 5: תמלול

זה השלב הכי איטי - תלוי באורך ההקלטה. עם GPU חינמי של Colab (T4), הקלטה של
3-4 שעות אמורה לקחת בסביבות שעה-שעתיים (במקום הרבה יותר על מעבד רגיל).
בזמן הריצה יודפסו קטעי תמלול בהדרגה כך שתוכלו לראות שזה מתקדם.


In [ ]:
import torch
from faster_whisper import WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
if device == "cpu":
    print("שימו לב: לא זוהה GPU. לכו ל-Runtime -> Change runtime type -> T4 GPU, "
          "ואז Runtime -> Restart session, כדי שזה ירוץ הרבה יותר מהר.")

print(f"טוען מודל תמלול על {device}...")
model = WhisperModel("ivrit-ai/whisper-large-v3-turbo-ct2", device=device, compute_type=compute_type)

segments_iter, info = model.transcribe(audio_filename, language="he", vad_filter=True)
segments = []
for seg in segments_iter:
    text = seg.text.strip()
    if not text:
        continue
    segments.append({"start": seg.start, "end": seg.end, "text": text})
    print(f"[{int(seg.end)}s / {int(info.duration)}s] {text}")

duration = info.duration
print(f"\nתמלול הושלם: {len(segments)} קטעים, אורך הקלטה כ-{duration/60:.0f} דקות")


## שלב 6: זיהוי דוברים (מי דיבר מתי)

In [ ]:
from pyannote.audio import Pipeline

print("טוען מודל זיהוי דוברים...")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", use_auth_token=HF_TOKEN
)
if device == "cuda":
    diarization_pipeline.to(torch.device("cuda"))

diarization = diarization_pipeline(audio_filename)
turns = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    turns.append({"start": turn.start, "end": turn.end, "speaker": speaker})

print(f"זוהו {len(set(t['speaker'] for t in turns))} דוברים")


## שלב 7: מיזוג התמלול עם זיהוי הדוברים

In [ ]:
def assign_speakers(transcript_segments, diarization_turns):
    if not diarization_turns:
        return [{**seg, "speaker": "SPEAKER_00"} for seg in transcript_segments]
    labeled = []
    for seg in transcript_segments:
        best_speaker, best_overlap = None, 0.0
        for turn in diarization_turns:
            overlap = min(seg["end"], turn["end"]) - max(seg["start"], turn["start"])
            if overlap > best_overlap:
                best_overlap, best_speaker = overlap, turn["speaker"]
        if best_speaker is None:
            nearest = min(
                diarization_turns,
                key=lambda t: min(abs(t["start"] - seg["end"]), abs(t["end"] - seg["start"])),
            )
            best_speaker = nearest["speaker"]
        labeled.append({**seg, "speaker": best_speaker})
    return labeled


def merge_consecutive(labeled_segments, max_gap_seconds=1.0):
    merged = []
    for seg in labeled_segments:
        prev = merged[-1] if merged else None
        if prev and prev["speaker"] == seg["speaker"] and seg["start"] - prev["end"] <= max_gap_seconds:
            prev["end"] = seg["end"]
            prev["text"] = f"{prev['text']} {seg['text']}".strip()
        else:
            merged.append(dict(seg))
    return merged


labeled = assign_speakers(segments, turns)
merged_segments = merge_consecutive(labeled)
speakers = sorted({s["speaker"] for s in merged_segments})
print(f"מוכן: {len(merged_segments)} קטעי דיבור, {len(speakers)} דוברים")


## שלב 8 (רשות): ניחוש שמות דוברים אוטומטי

In [ ]:
speaker_names = {}
if ANTHROPIC_API_KEY.strip():
    import json
    import anthropic

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    transcript_text = "\n".join(f'{s["speaker"]}: {s["text"]}' for s in merged_segments)[:300000]
    prompt = (
        "להלן תמלול אוטומטי של שיחה בעברית, מחולק לפי דוברים. תוויות הדוברים "
        "(SPEAKER_00 וכו') הן זיהוי קולי אוטומטי בלבד. זהה את השם האמיתי של כל "
        'דובר רק אם יש רמז ברור בטקסט (הצגה עצמית, פנייה בשם). החזר אך ורק JSON '
        'תקין: {"SPEAKER_00": "שם או null", ...}\n\n' + transcript_text
    )
    response = client.messages.create(
        model="claude-sonnet-5", max_tokens=1024, messages=[{"role": "user", "content": prompt}]
    )
    raw = "".join(b.text for b in response.content if getattr(b, "type", "") == "text").strip().strip("`")
    if raw.startswith("json"):
        raw = raw[4:].strip()
    try:
        guessed = json.loads(raw)
        speaker_names = {
            sp: guessed[sp].strip()
            for sp in speakers
            if isinstance(guessed.get(sp), str) and guessed[sp].strip() and guessed[sp].strip().lower() != "null"
        }
    except Exception as exc:
        print(f"ניחוש השמות נכשל ({exc}) - הדוברים יישארו עם תוויות ברירת מחדל.")

print(f"שמות שזוהו: {speaker_names if speaker_names else 'לא זוהו שמות - יישארו דובר 1, דובר 2 וכו׳'}")


## שלב 9 (רשות): תיקון/הוספת שמות דוברים ידנית

אם השם שזוהה אוטומטית שגוי, או שרוצים להוסיף שם שלא זוהה - אפשר לערוך כאן
ידנית ולהריץ את התא הזה שוב. לדוגמה (מחקו את ה-# כדי להפעיל):

```
# speaker_names["SPEAKER_00"] = "יוסי כהן"
# speaker_names["SPEAKER_01"] = "דנה לוי"
```


In [ ]:
# ערכו כאן במידת הצורך, לדוגמה:
# speaker_names["SPEAKER_00"] = "יוסי כהן"

for i, sp in enumerate(speakers):
    print(f"{sp} -> {speaker_names.get(sp, f'דובר {i + 1}')}")


## שלב 10: ייצוא התמלול והורדה (TXT / SRT / Word)

In [ ]:
from google.colab import files


def speaker_display_name(speaker, names, all_speakers):
    custom = (names.get(speaker) or "").strip()
    if custom:
        return custom
    idx = all_speakers.index(speaker) if speaker in all_speakers else -1
    return f"דובר {idx + 1}" if idx >= 0 else speaker


def format_ts_short(total_seconds):
    total_seconds = max(0, int(total_seconds))
    hours, rem = divmod(total_seconds, 3600)
    minutes, seconds = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def format_srt_ts(total_seconds):
    total_seconds = max(0.0, total_seconds)
    hours, rem = divmod(total_seconds, 3600)
    minutes, seconds = divmod(rem, 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    return f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d},{millis:03d}"


base_name = audio_filename.rsplit(".", 1)[0]

# ===== TXT =====
txt_lines = [f"תמלול: {audio_filename}", ""]
for seg in merged_segments:
    name = speaker_display_name(seg["speaker"], speaker_names, speakers)
    txt_lines.append(f"[{format_ts_short(seg['start'])}] {name}: {seg['text']}")
txt_path = f"{base_name}.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(txt_lines))

# ===== SRT =====
srt_blocks = []
for i, seg in enumerate(merged_segments, start=1):
    name = speaker_display_name(seg["speaker"], speaker_names, speakers)
    srt_blocks.append(
        f"{i}\n{format_srt_ts(seg['start'])} --> {format_srt_ts(seg['end'])}\n{name}: {seg['text']}\n"
    )
srt_path = f"{base_name}.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(srt_blocks))

# ===== DOCX =====
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement


def set_rtl(paragraph):
    p_pr = paragraph._p.get_or_add_pPr()
    bidi = OxmlElement("w:bidi")
    p_pr.append(bidi)


doc = Document()
title = doc.add_heading(f"תמלול: {audio_filename}", level=1)
title.alignment = WD_ALIGN_PARAGRAPH.RIGHT
set_rtl(title)
for seg in merged_segments:
    name = speaker_display_name(seg["speaker"], speaker_names, speakers)
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    set_rtl(p)
    run1 = p.add_run(f"[{format_ts_short(seg['start'])}] {name}: ")
    run1.bold = True
    p.add_run(seg["text"])
docx_path = f"{base_name}.docx"
doc.save(docx_path)

print("מוריד קבצים...")
files.download(txt_path)
files.download(srt_path)
files.download(docx_path)
print("סיימנו! בדקו את תיקיית ההורדות של הדפדפן.")
